
# Disruptive Development Path Measurement via Patent Citation Network
Replicação de: Wang et al. (2024), Journal of Informetrics 18, 101493
DOI: https://doi.org/10.1016/j.joi.2024.101493

## Pipeline completo:
  1. Carregar e preparar dados de citação de patentes
  2. Construir rede de citação 2-012U
  3. Classificar triplets (5-021U, 6-021C, 9-030T)
  4. Calcular grau de disrupção D' (Fórmula 3)
  5. Contrair a rede por threshold de disrupção
  6. Extrair caminho principal via SPLC
  7. Extrair caminho crítico via CPM
  8. Visualizar resultados

## Instalar Dependências:
  pip install pandas networkx matplotlib seaborn openpyxl


In [1]:
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from typing import Optional, Dict, List, Tuple, Set
import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
def load_bigquery_json(json_metadata_path: str, json_network_path: str) -> Tuple[pd.DataFrame, dict]:
    """
    Lê os arquivos JSONL exportados das duas queries do BigQuery e os
    transforma no formato de pares e no mapa de anos esperados pelo pipeline.

    Notas sobre os dados:
    - priority_date chega como string "YYYYMMDD" (p.ex. "20080905")
    - O arquivo de rede traz citing_date (data do citante), MAS NÃO a data
      do patent citado. Por isso, year_map dos focais é essencial para cobrir
      os cited_patent que são patentes BR focais.
    - Citações backward de focais → patentes internacionais terão year_cited=NaN
      (data da citada não está na base) e serão descartadas automaticamente.
    """
    # 1. Carregar os Metadados (Para montar o year_map)
    print(f"[BigQuery] Carregando metadados de {json_metadata_path}...")
    df_meta = pd.read_json(json_metadata_path, lines=True, dtype={"priority_date": str})

    # priority_date vem como "YYYYMMDD" — converter para ano inteiro
    df_meta['year'] = pd.to_datetime(
        df_meta['priority_date'].astype(str), format='%Y%m%d', errors='coerce'
    ).dt.year

    year_map = dict(zip(df_meta['publication_number'], df_meta['year']))
    print(f"[BigQuery] {len(year_map):,} patentes focais no year_map "
          f"(anos {int(df_meta['year'].min())}–{int(df_meta['year'].max())})")

    # 2. Carregar a Rede de Citações Global
    print(f"[BigQuery] Carregando rede de citações de {json_network_path}...")
    df_net = pd.read_json(json_network_path, lines=True, dtype={"citing_date": str})

    # citing_date também é "YYYYMMDD"
    df_net['citing_year'] = pd.to_datetime(
        df_net['citing_date'].astype(str), format='%Y%m%d', errors='coerce'
    ).dt.year

    # Adicionar anos dos citantes ao year_map (sem sobrescrever os focais)
    new_years = dict(zip(df_net['citing_patent'], df_net['citing_year']))
    year_map = {**new_years, **year_map}   # focais têm prioridade

    # 3. Preparar df_pairs
    df_pairs = df_net[['citing_patent', 'cited_patent']].copy()
    df_pairs.columns = ['citing', 'cited']

    df_pairs['year_citing'] = df_pairs['citing'].map(year_map)
    df_pairs['year_cited']  = df_pairs['cited'].map(year_map)

    n_before = len(df_pairs)
    df_pairs = df_pairs.dropna(subset=['year_citing', 'year_cited'])
    n_dropped = n_before - len(df_pairs)

    # Diagnóstico: verificar se há citações backward (BR focal → qualquer)
    focal_set = set(df_meta['publication_number'])
    backward_rows = df_pairs[df_pairs['citing'].isin(focal_set)]
    forward_rows  = df_pairs[df_pairs['cited'].isin(focal_set)]

    print(f"[BigQuery] {len(df_pairs):,} pares válidos "
          f"({n_dropped:,} descartados por year_cited ausente)")
    print(f"  → Backward (focal→?): {len(backward_rows):,} | "
          f"Forward (?→focal): {len(forward_rows):,}")
    if len(backward_rows) == 0:
        print("  ⚠ AVISO: Sem citações backward registradas para as patentes focais.")
        print("    Isso impede o cálculo de D'. A rede terá apenas citações forward.")
        print("    Solução: re-executar Query 2 no BigQuery incluindo citada.priority_date.")

    return df_pairs, year_map


In [ ]:
# =============================================================================
# ETAPA 1: DADOS DE ENTRADA
# =============================================================================

def create_synthetic_example() -> Tuple[pd.DataFrame, dict]:
    """
    Gera dados sintéticos para testar o pipeline sem arquivo externo.
    Simula uma pequena rede de sintering technology com ~30 patentes.
    
    Retorna
    -------
    df_pairs : DataFrame com pares citing→cited
    year_map : dict {patent_id: year}
    """
    # Patentes com seus anos (simulação de caminho principal + ramificações)
    patents = {
        "P1995": 1995, "P1996a": 1996, "P1996b": 1996,
        "P1998": 1998, "P1999": 1999,
        "P2001": 2001, "P2002a": 2002, "P2002b": 2002,
        "P2004": 2004, "P2005a": 2005, "P2005b": 2005,
        "P2007": 2007, "P2008a": 2008, "P2008b": 2008, "P2008c": 2008,
        "P2010": 2010, "P2011a": 2011, "P2011b": 2011,
        "P2012": 2012, "P2013a": 2013, "P2013b": 2013, "P2013c": 2013,
        "P2015a": 2015, "P2015b": 2015, "P2015c": 2015,
        "P2016": 2016, "P2017a": 2017, "P2017b": 2017,
        "P2018": 2018, "P2019": 2019
    }
    
    # Pares de citação (mais recente cita mais antigo)
    citation_pairs = [
        # Linha principal (caminho disruptivo)
        ("P1996a", "P1995"),
        ("P1998", "P1996a"), ("P1998", "P1996b"),
        ("P1999", "P1998"),
        ("P2001", "P1999"),
        ("P2002a", "P2001"),
        ("P2004", "P2002a"),
        ("P2005a", "P2004"),
        ("P2007", "P2005a"),
        ("P2008a", "P2007"),
        ("P2010", "P2008a"),
        ("P2012", "P2010"),
        ("P2016", "P2012"),
        ("P2018", "P2016"),
        ("P2019", "P2018"),
        # Ramificações (co-citações / subáreas)
        ("P1996b", "P1995"),
        ("P2002b", "P1999"), ("P2002b", "P1998"),  # co-citação
        ("P2005b", "P2002a"), ("P2005b", "P2002b"),
        ("P2008b", "P2005a"), ("P2008b", "P2005b"),
        ("P2008c", "P2004"), ("P2008c", "P2005a"),  # 9-030T: cita focal e backward
        ("P2011a", "P2008a"), ("P2011a", "P2007"),  # 9-030T
        ("P2011b", "P2008b"),
        ("P2013a", "P2011a"), ("P2013a", "P2010"),  # 9-030T
        ("P2013b", "P2011b"),
        ("P2013c", "P2010"),
        ("P2015a", "P2013a"),
        ("P2015b", "P2013b"), ("P2015b", "P2013a"),
        ("P2015c", "P2012"), ("P2015c", "P2010"),   # 9-030T
        ("P2017a", "P2015a"), ("P2017a", "P2016"),
        ("P2017b", "P2015b"),
    ]
    
    df_pairs = pd.DataFrame(citation_pairs, columns=["citing", "cited"])
    df_pairs["year_citing"] = df_pairs["citing"].map(patents)
    df_pairs["year_cited"] = df_pairs["cited"].map(patents)
    
    print("[Sintético] Rede gerada com "
          f"{len(df_pairs)} pares de citação e "
          f"{len(patents)} patentes")
    return df_pairs, patents

In [ ]:
# =============================================================================
# ETAPA 2: CONSTRUÇÃO DA REDE DE CITAÇÃO (2-012U)
# =============================================================================

def build_citation_network(df_pairs: pd.DataFrame,
                           year_map: dict,
                           core_patents: Optional[Set] = None) -> nx.DiGraph:
    """
    Constrói a rede 2-012U: dígrafo onde aresta (A → B) significa "A cita B".
    
    A direção da citação é oposta ao fluxo de conhecimento, conforme o paper
    (seção 3.1.1): conhecimento flui de B para A.

    Parâmetros
    ----------
    df_pairs : DataFrame com colunas ['citing', 'cited']
    year_map : dict {patent_id: year}
    core_patents : conjunto de patentes core (filtra não-core se fornecido)

    Retorna
    -------
    G : DiGraph com atributo 'year' em cada nó
    """
    G = nx.DiGraph()
    
    for _, row in df_pairs.iterrows():
        citing, cited = row["citing"], row["cited"]
        
        # Filtrar citações a patentes não-core
        if core_patents is not None:
            if citing not in core_patents or cited not in core_patents:
                continue
        
        # Garantir temporalidade: citante deve ser posterior ao citado
        y_citing = year_map.get(citing, 0)
        y_cited  = year_map.get(cited, 0)
        if y_citing <= y_cited:
            continue  # remove anomalias de data (seção 3.4.1 do paper)
        
        G.add_node(citing, year=y_citing)
        G.add_node(cited,  year=y_cited)
        G.add_edge(citing, cited)  # citing → cited

    print(f"[2-012U] Nós: {G.number_of_nodes():,} | Arestas: {G.number_of_edges():,} | "
          f"DAG: {nx.is_directed_acyclic_graph(G)}")
    return G


def remove_self_citations(G: nx.DiGraph, patent_families: Optional[dict] = None) -> nx.DiGraph:
    """
    Remove auto-citações dentro da mesma família de patentes (seção 3.2.3 do paper).
    
    patent_families : dict {patent_id: family_id}
    """
    if patent_families is None:
        return G
    
    removed = 0
    edges_to_remove = []
    for u, v in G.edges():
        if patent_families.get(u) == patent_families.get(v):
            edges_to_remove.append((u, v))
            removed += 1
    
    G.remove_edges_from(edges_to_remove)
    print(f"[Auto-citações] {removed} arestas removidas")
    return G

In [ ]:
# =============================================================================
# ETAPA 3: CLASSIFICAÇÃO DE TRIPLETS E CÁLCULO DA DISRUPÇÃO (D')
# =============================================================================

def classify_triplets_and_disruption(
        G: nx.DiGraph,
        year_map: dict,
        focal_patents: Optional[List] = None) -> pd.DataFrame:
    """
    Para cada patente focal, classifica as citações forward em tipos i, j, k
    e calcula o grau de disrupção D' usando a Fórmula 3 do paper.

    Definições (Fig. 4 do paper):
      - Tipo i (disruptivo): cita APENAS a patente focal, NÃO suas backward citations
      - Tipo j (não-disruptivo / 9-030T): cita a focal E pelo menos uma backward
      - Tipo k (consolidador): cita backward citations mas NÃO a focal

    Fórmula 3: D' = ni / (ni + ñk)
    onde ñk = nj + nk (todas as patentes que citam qualquer backward da focal)

    Por que Fórmula 3 e não 1?
      Formula 1: D = (ni-nj)/(ni+nj+nk) — subtrai j de i, o que implica que
        i contém j (incorreto por definição). A Fórmula 3 corrige isso.

    Parâmetros
    ----------
    G : DiGraph da rede 2-012U
    year_map : dict {patent_id: year}
    focal_patents : lista de patentes a analisar (default: todas com in+out grau > 0)

    Retorna
    -------
    DataFrame com colunas: patent, ni, nj, nk, nk_tilde, d_prime, year, ...
    """
    if focal_patents is None:
        focal_patents = [
            n for n in G.nodes()
            if G.in_degree(n) > 0 and G.out_degree(n) > 0
        ]

    results = []

    for focal in focal_patents:
        focal_year = year_map.get(focal, 0)

        # Backward citations: patentes que a focal cita (focal → backward) 
        backward: Set = set(G.successors(focal))
        if not backward:
            continue

        # Forward citations: patentes que citam a focal, com ano posterior 
        forward: Set = {
            n for n in G.predecessors(focal)
            if year_map.get(n, 0) > focal_year
        }
        if not forward:
            continue

        # Classificação i vs j 
        ni = 0   # forward que cita APENAS focal
        nj = 0   # forward que cita focal E alguma backward (9-030T)
        
        for fwd in forward:
            fwd_cites: Set = set(G.successors(fwd))  # o que fwd cita
            if fwd_cites & backward:                  # interseção com backward
                nj += 1  # tipo j: transitive + co-citation → 9-030T
            else:
                ni += 1  # tipo i: apenas transitive → 6-021C puro

        #  Tipo k: citam backward mas NÃO citam a focal 
        # Encontra todos os citantes de qualquer backward com ano posterior ao focal
        all_backward_citers: Set = set()
        for back in backward:
            all_backward_citers.update(
                n for n in G.predecessors(back)
                if year_map.get(n, 0) > focal_year
            )
        
        # k_tilde (ñk) = todos que citam alguma backward (= j + k)
        nk_tilde = len(all_backward_citers)
        nk = nk_tilde - nj  # tipo k puro: backward citers que NÃO citam focal

        #  Fórmula 3 
        if ni + nk_tilde == 0:
            d_prime = np.nan
        else:
            d_prime = ni / (ni + nk_tilde)

        #  Fórmula 1 original (Wu et al., 2019) para comparação 
        denom = ni + nj + nk
        d_original = (ni - nj) / denom if denom > 0 else np.nan

        results.append({
            "patent"         : focal,
            "year"           : focal_year,
            "ni"             : ni,
            "nj"             : nj,
            "nk"             : nk,
            "nk_tilde"       : nk_tilde,
            "n_forward"      : len(forward),
            "n_backward"     : len(backward),
            "d_prime"        : d_prime,    # Fórmula 3 (melhorada)
            "d_original"     : d_original, # Fórmula 1 (Wu et al., 2019)
            "network_type"   : _classify_network_type(ni, nj, nk),
        })

    df_result = pd.DataFrame(results).sort_values("year").reset_index(drop=True)
    
    # Estatísticas resumidas
    valid = df_result["d_prime"].dropna()
    print(f"\n[Disrupção D'] Calculada para {len(df_result):,} patentes focais")
    print(f"  Média: {valid.mean():.3f} | Mediana: {valid.median():.3f} | "
          f"Std: {valid.std():.3f}")
    print(f"  Faixa: [{valid.min():.3f}, {valid.max():.3f}]")
    print(f"  D'>0.5: {(valid>0.5).sum()} | D'<0.2: {(valid<0.2).sum()}")
    
    return df_result


def _classify_network_type(ni: int, nj: int, nk: int) -> str:
    """Classifica a patente focal pelo tipo de rede dominante."""
    if ni > 0 and nj == 0:
        return "6-021C"    # apenas transitive citation (disruptivo puro)
    elif ni == 0 and nj > 0:
        return "9-030T"    # apenas transitive+co-citation (consolidador)
    elif ni > 0 and nj > 0:
        return "Mixed"
    else:
        return "5-021U"    # apenas co-citation

In [ ]:
# =============================================================================
# ETAPA 4: CONTRAÇÃO DA REDE (REMOÇÃO DE 9-030T)
# =============================================================================

def contract_citation_network(
        G: nx.DiGraph,
        df_disruption: pd.DataFrame,
        disruption_threshold: float = 0.3,
        min_citations: int = 2) -> Tuple[nx.DiGraph, pd.DataFrame]:
    """
    Contrai a rede de citação em dois passos (seção 3.3.2 do paper):

    Passo 1 — Filtragem topológica:
      Mantém apenas arestas onde a patente focal tem APENAS relação transitive
      (6-021C sem 9-030T), ou seja, tipo "6-021C" ou "Mixed" com D' alto.

    Passo 2 — Threshold de disrupção:
      Remove nós com D' abaixo do threshold definido.

    Parâmetros
    ----------
    G : DiGraph original (2-012U)
    df_disruption : DataFrame com colunas ['patent', 'd_prime', 'network_type']
    disruption_threshold : float — mínimo de D' para manter o nó
    min_citations : int — mínimo de citações forward para incluir no caminho

    Retorna
    -------
    G_contracted : DiGraph contraído
    df_kept : DataFrame das patentes mantidas
    """
    # Identificar patentes que passam no threshold
    df_valid = df_disruption[
        (df_disruption["d_prime"] >= disruption_threshold) &
        (df_disruption["n_forward"] >= min_citations)
    ].copy()
    
    valid_patents = set(df_valid["patent"])
    
    # Construir subgrafo com nós válidos
    G_contracted = G.subgraph(valid_patents).copy()
    
    # Remover possíveis ciclos (não deve haver em DAG, mas garantir)
    if not nx.is_directed_acyclic_graph(G_contracted):
        print("[Aviso] Ciclos detectados! Removendo...")
        G_contracted = _remove_cycles(G_contracted)
    
    print(f"\n[Contração] Threshold D' ≥ {disruption_threshold}")
    print(f"  Original: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")
    print(f"  Contraída: {G_contracted.number_of_nodes()} nós, "
          f"{G_contracted.number_of_edges()} arestas")
    print(f"  Redução: {(1 - G_contracted.number_of_nodes()/G.number_of_nodes())*100:.1f}%")
    
    return G_contracted, df_valid


def _remove_cycles(G: nx.DiGraph) -> nx.DiGraph:
    """Remove arestas mínimas para tornar o grafo acíclico (feedback arc set)."""
    G_copy = G.copy()
    while not nx.is_directed_acyclic_graph(G_copy):
        cycles = list(nx.simple_cycles(G_copy))
        if not cycles:
            break
        cycle = cycles[0]
        # Remove a aresta com menor peso no ciclo
        G_copy.remove_edge(cycle[-1], cycle[0])
    return G_copy

In [ ]:
# =============================================================================
# ETAPA 5: EXTRAÇÃO DO CAMINHO PRINCIPAL (SPLC)
# =============================================================================

def compute_splc_weights(G: nx.DiGraph) -> Dict[Tuple, float]:
    """
    Calcula pesos SPLC (Search Path Link Count) para cada aresta.

    SPLC(u → v) = (nº de caminhos de qualquer fonte até u) ×
                  (nº de caminhos de v até qualquer sorvedouro)

    O SPLC atribui maior peso a arestas que conectam muitos pares
    fonte-sorvedouro, identificando as ligações mais "críticas" da rede.

    Parâmetros
    ----------
    G : DiGraph acíclico (DAG)

    Retorna
    -------
    edge_weights : dict {(u, v): splc_weight}
    paths_to     : dict {node: nº caminhos até o nó}
    paths_from   : dict {node: nº caminhos a partir do nó}
    """
    topo_order = list(nx.topological_sort(G))

    #  Forward pass: caminhos de fontes até cada nó 
    paths_to = {n: 0 for n in G.nodes()}
    for n in topo_order:
        if G.in_degree(n) == 0:          # nó fonte
            paths_to[n] = 1
        else:
            paths_to[n] = sum(paths_to[p] for p in G.predecessors(n))

    #  Backward pass: caminhos de cada nó até sorvedouros 
    paths_from = {n: 0 for n in G.nodes()}
    for n in reversed(topo_order):
        if G.out_degree(n) == 0:          # nó sorvedouro
            paths_from[n] = 1
        else:
            paths_from[n] = sum(paths_from[s] for s in G.successors(n))

    #  Peso SPLC por aresta 
    edge_weights = {
        (u, v): paths_to[u] * paths_from[v]
        for u, v in G.edges()
    }

    max_w = max(edge_weights.values()) if edge_weights else 1
    print(f"\n[SPLC] Pesos calculados | Max: {max_w} | "
          f"Arestas com peso > 0: {sum(1 for w in edge_weights.values() if w>0)}")

    return edge_weights, paths_to, paths_from


def extract_main_path(G: nx.DiGraph,
                      edge_weights: Dict,
                      year_map: dict,
                      disruption_df: Optional[pd.DataFrame] = None) -> List:
    """
    Extrai o caminho principal via busca padrão global (Standard Search).
    
    Algoritmo: programação dinâmica sobre ordem topológica.
    Maximiza a soma dos pesos SPLC ao longo do caminho.

    Parâmetros
    ----------
    G : DAG com arestas ponderadas
    edge_weights : dict com pesos SPLC
    year_map : dict para ordenar nós por ano
    disruption_df : opcional, para anotar nós no resultado

    Retorna
    -------
    main_path : lista de patentes na ordem do caminho principal
    """
    if G.number_of_nodes() == 0:
        print("[ERRO] Rede vazia após contração. Reduza o threshold.")
        return []

    topo_order = list(nx.topological_sort(G))

    # Programação dinâmica: max peso acumulado até cada nó
    max_weight   = {n: 0.0 for n in G.nodes()}
    predecessor  = {n: None for n in G.nodes()}

    for v in topo_order:
        for u in G.predecessors(v):
            w = edge_weights.get((u, v), 0)
            if max_weight[u] + w > max_weight[v]:
                max_weight[v]  = max_weight[u] + w
                predecessor[v] = u

    # Encontrar o melhor sorvedouro (nó sem sucessores)
    sinks = [n for n in G.nodes() if G.out_degree(n) == 0]
    if not sinks:
        sinks = list(G.nodes())
    
    best_sink = max(sinks, key=lambda s: max_weight[s])

    # Reconstruir caminho por retrocesso
    path = []
    node = best_sink
    while node is not None:
        path.append(node)
        node = predecessor[node]
    path = list(reversed(path))

    # Anotar com anos e disrupção
    if disruption_df is not None:
        d_map = dict(zip(disruption_df["patent"], disruption_df["d_prime"]))
    else:
        d_map = {}

    print(f"\n[Caminho Principal] {len(path)} nós")
    print("  " + " → ".join(
        f"{n}({year_map.get(n,'?')}, D'={d_map.get(n, float('nan')):.2f})"
        for n in path
    ))

    return path

In [ ]:
# =============================================================================
# ETAPA 6: CAMINHO CRÍTICO (CPM — Critical Path Method)
# =============================================================================

def extract_critical_path(G: nx.DiGraph,
                          edge_weights: Dict,
                          year_map: dict) -> Tuple[List, nx.DiGraph]:
    """
    Extrai o caminho crítico usando CPM (seção 3.3.3 e 4.4 do paper).
    
    O caminho crítico é o caminho mais longo (em soma de pesos) da rede,
    além do caminho principal. Inclui nós-chave e ramificações importantes.

    Usa longest-path em DAG via programação dinâmica.

    Retorna
    -------
    critical_path : lista de nós no caminho crítico
    G_critical : subgrafo do caminho crítico com todos os predecessores
    """
    if G.number_of_nodes() == 0:
        return [], nx.DiGraph()

    topo_order = list(nx.topological_sort(G))

    #  Forward pass para CPM 
    early_start  = {n: 0 for n in G.nodes()}
    for n in topo_order:
        for s in G.successors(n):
            w = edge_weights.get((n, s), 1)
            early_start[s] = max(early_start[s], early_start[n] + w)

    # O caminho crítico termina no nó com maior early_start
    end_node = max(early_start, key=early_start.get)

    #  Backward pass 
    late_start = {n: early_start[end_node] for n in G.nodes()}
    for n in reversed(topo_order):
        for p in G.predecessors(n):
            w = edge_weights.get((p, n), 1)
            late_start[p] = min(late_start[p], late_start[n] - w)

    # Identificar nós e arestas críticas (slack = 0)
    critical_nodes = set()
    critical_edges = []
    for u, v in G.edges():
        w = edge_weights.get((u, v), 1)
        slack = late_start[v] - early_start[u] - w
        if abs(slack) < 1e-9:
            critical_nodes.add(u)
            critical_nodes.add(v)
            critical_edges.append((u, v))

    G_critical = G.subgraph(critical_nodes).copy()
    
    # Reconstruir sequência do caminho crítico
    critical_path = [
        n for n in topo_order if n in critical_nodes
    ]

    print(f"\n[CPM] Caminho crítico: {len(critical_nodes)} nós, "
          f"{len(critical_edges)} arestas críticas")

    return critical_path, G_critical

In [ ]:
# =============================================================================
# ETAPA 7: ANÁLISE ANUAL DO GRAU DE DISRUPÇÃO
# =============================================================================

def analyze_annual_disruption(df_disruption: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega o grau de disrupção por ano para identificar ondas de inovação.
    
    Replicação da Fig. 8 do paper.
    
    Retorna
    -------
    df_annual : DataFrame com colunas year, count, mean_d, median_d, std_d
    """
    df_annual = (df_disruption
        .dropna(subset=["d_prime"])
        .groupby("year")
        .agg(
            count    = ("patent", "count"),
            mean_d   = ("d_prime", "mean"),
            median_d = ("d_prime", "median"),
            std_d    = ("d_prime", "std"),
            max_d    = ("d_prime", "max"),
        )
        .reset_index()
    )
    
    # Interpretação (conforme seção 4.2 do paper):
    # D' próximo de 0.5 com dados suficientes → tecnologia matura
    # Picos em D' → possíveis novas ondas disruptivas
    df_annual["interpretation"] = df_annual.apply(
        lambda r: "⚡ Pico disruptivo" if r["mean_d"] > 0.5 and r["count"] >= 5
        else ("📈 Alta disrupção" if r["mean_d"] > 0.4 else ""),
        axis=1
    )
    
    return df_annual

In [ ]:
# =============================================================================
# ETAPA 8: VISUALIZAÇÕES
# =============================================================================

def plot_disruption_annual(df_annual: pd.DataFrame, 
                           title: str = "Grau de Disrupção Anual",
                           save_path: Optional[str] = None):
    """Replicação da Fig. 8 do paper: quantidade de patentes + D' médio por ano."""
    fig, ax1 = plt.subplots(figsize=(12, 5))
    
    color_bar  = "#4C72B0"
    color_line = "#DD8452"
    
    ax1.bar(df_annual["year"], df_annual["count"],
            color=color_bar, alpha=0.7, label="Qtd. patentes focais")
    ax1.set_xlabel("Ano", fontsize=11)
    ax1.set_ylabel("Quantidade de patentes focais", color=color_bar, fontsize=11)
    ax1.tick_params(axis="y", labelcolor=color_bar)
    
    ax2 = ax1.twinx()
    ax2.plot(df_annual["year"], df_annual["mean_d"],
             color=color_line, linewidth=2.5, marker="o",
             markersize=5, label="D' médio")
    ax2.fill_between(df_annual["year"],
                     df_annual["mean_d"] - df_annual["std_d"].fillna(0),
                     df_annual["mean_d"] + df_annual["std_d"].fillna(0),
                     color=color_line, alpha=0.15)
    ax2.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, linewidth=1)
    ax2.set_ylabel("Grau de disrupção médio (D')", color=color_line, fontsize=11)
    ax2.tick_params(axis="y", labelcolor=color_line)
    ax2.set_ylim(0, 1.1)
    
    # Anotar picos
    for _, row in df_annual[df_annual["mean_d"] > 0.5].iterrows():
        ax2.annotate(f"{row['mean_d']:.2f}",
                     xy=(row["year"], row["mean_d"]),
                     xytext=(0, 8), textcoords="offset points",
                     ha="center", fontsize=8, color=color_line)
    
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    
    plt.title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Fig] Salvo em: {save_path}")
    plt.show()


def _hierarchical_layout_by_year(G: nx.DiGraph, year_map: dict) -> dict:
    """Layout vertical: eixo Y = ano (mais antigo em cima), X = posição na camada."""
    years = sorted(set(year_map.get(n, 0) for n in G.nodes()))
    year_nodes = defaultdict(list)
    for n in G.nodes():
        year_nodes[year_map.get(n, 0)].append(n)
    
    pos = {}
    y_scale = -1.0  # anos crescentes para baixo
    for y_idx, year in enumerate(years):
        nodes_in_year = year_nodes[year]
        n_nodes = len(nodes_in_year)
        for x_idx, node in enumerate(nodes_in_year):
            x = (x_idx - (n_nodes - 1) / 2) * 1.5
            pos[node] = (x, y_idx * y_scale)
    return pos


def plot_main_path(G: nx.DiGraph,
                   main_path: List,
                   year_map: dict,
                   disruption_df: Optional[pd.DataFrame] = None,
                   title: str = "Caminho Principal",
                   save_path: Optional[str] = None):
    """
    Visualiza o caminho principal na rede de citação.
    Nós do caminho principal são destacados.
    Replicação da Fig. 9(a) do paper.
    """
    if len(G.nodes()) == 0:
        print("[Aviso] Grafo vazio, nada para visualizar.")
        return

    fig, ax = plt.subplots(figsize=(10, 12))
    
    # Layout hierárquico por ano
    pos = _hierarchical_layout_by_year(G, year_map)
    
    # Cores: caminho principal = laranja, resto = azul claro
    main_path_set = set(main_path)
    node_colors = [
        "#E8623C" if n in main_path_set else "#7FB3D3"
        for n in G.nodes()
    ]
    node_sizes = [
        400 if n in main_path_set else 150
        for n in G.nodes()
    ]
    
    # Arestas: caminho principal = laranja escuro, resto = cinza
    edge_colors = []
    edge_widths = []
    for u, v in G.edges():
        if u in main_path_set and v in main_path_set:
            edge_colors.append("#C0392B")
            edge_widths.append(2.5)
        else:
            edge_colors.append("#AAAAAA")
            edge_widths.append(0.8)
    
    nx.draw_networkx(
        G, pos=pos, ax=ax,
        node_color=node_colors,
        node_size=node_sizes,
        edge_color=edge_colors,
        width=edge_widths,
        labels={n: n for n in G.nodes()},
        font_size=6,
        arrows=True,
        arrowsize=10,
    )
    
    legend_elements = [
        mpatches.Patch(color="#E8623C", label="Caminho principal"),
        mpatches.Patch(color="#7FB3D3", label="Outros nós"),
    ]
    ax.legend(handles=legend_elements, loc="upper right")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.axis("off")
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Fig] Salvo em: {save_path}")
    plt.show()


def plot_disruption_distribution(df_disruption: pd.DataFrame,
                                 save_path: Optional[str] = None):
    """Histograma da distribuição de D' + comparação com D original."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    df_valid = df_disruption.dropna(subset=["d_prime"])
    
    axes[0].hist(df_valid["d_prime"], bins=20, color="#4C72B0", 
                 edgecolor="white", alpha=0.85)
    axes[0].axvline(x=0.5, color="red", linestyle="--", alpha=0.7)
    axes[0].set_xlabel("D' (Fórmula 3 — melhorada)")
    axes[0].set_ylabel("Frequência")
    axes[0].set_title("Distribuição do Grau de Disrupção D'")
    
    df_both = df_disruption.dropna(subset=["d_prime", "d_original"])
    axes[1].scatter(df_both["d_original"], df_both["d_prime"],
                    alpha=0.4, s=20, color="#DD8452")
    axes[1].plot([-1, 1], [-1, 1], "r--", alpha=0.4)
    axes[1].set_xlabel("D (Fórmula 1 original — Wu et al., 2019)")
    axes[1].set_ylabel("D' (Fórmula 3 melhorada)")
    axes[1].set_title("Comparação: D original vs D' melhorado")
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# =============================================================================
# PIPELINE COMPLETO
# =============================================================================

def run_full_pipeline(
        data_source: str = "synthetic",
        filepath: Optional[str] = None,
        json_metadata_path: Optional[str] = None,
        json_network_path: Optional[str] = None,
        disruption_threshold: float = 0.3,
        min_citations: int = 2,
        save_figures: bool = False,
        output_dir: str = ".") -> dict:
    """
    Executa o pipeline completo de medição do caminho disruptivo.
    """
    print("=" * 60)
    print("PIPELINE: CAMINHO DE DESENVOLVIMENTO DISRUPTIVO")
    print("Wang et al. (2024), Journal of Informetrics 18, 101493")
    print("=" * 60)

    #  1. Dados ─
    print("\n[1/7] Carregando dados...")
    if data_source == "synthetic":
        df_pairs, year_map = create_synthetic_example()
    elif data_source == "bigquery" and json_metadata_path and json_network_path:
        df_pairs, year_map = load_bigquery_json(json_metadata_path, json_network_path)
    else:
        raise ValueError("Especifique corretamente a origem dos dados (synthetic, file ou bigquery) e seus caminhos.")

    #  2. Rede 2-012U 
    print("\n[2/7] Construindo rede 2-012U...")
    G = build_citation_network(df_pairs, year_map)

    #  3. Grau de disrupção ─
    print("\n[3/7] Classificando triplets e calculando D'...")
    df_disruption = classify_triplets_and_disruption(G, year_map)

    # 4. Análise anual ─
    print("\n[4/7] Análise temporal...")
    df_annual = analyze_annual_disruption(df_disruption)
    print(df_annual.to_string(index=False))

    #  5. Contração 
    print("\n[5/7] Contraindo rede...")
    G_contracted, df_kept = contract_citation_network(
        G, df_disruption, disruption_threshold, min_citations
    )

    if G_contracted.number_of_nodes() == 0:
        print("[AVISO] Rede contraída vazia. Reduzindo threshold para 0.1...")
        G_contracted, df_kept = contract_citation_network(
            G, df_disruption, 0.1, 1
        )

    #  6. SPLC + Caminho principal ─
    print("\n[6/7] Calculando SPLC e extraindo caminho principal...")
    edge_weights, paths_to, paths_from = compute_splc_weights(G_contracted)
    main_path = extract_main_path(G_contracted, edge_weights, year_map, df_disruption)

    #  7. Caminho crítico (CPM) 
    print("\n[7/7] Extraindo caminho crítico (CPM)...")
    critical_path, G_critical = extract_critical_path(
        G_contracted, edge_weights, year_map
    )

    #  Visualizações 
    print("\n[Viz] Gerando visualizações...")
    
    save_disrupt = f"{output_dir}/fig_disruption_annual.png" if save_figures else None
    save_path    = f"{output_dir}/fig_main_path.png"        if save_figures else None
    save_dist    = f"{output_dir}/fig_distribution.png"     if save_figures else None
    
    plot_disruption_annual(df_annual, save_path=save_disrupt)
    plot_main_path(G_contracted, main_path, year_map, df_disruption, 
                   save_path=save_path)
    plot_disruption_distribution(df_disruption, save_path=save_dist)

    #  Resumo final 
    print("\n" + "=" * 60)
    print("RESUMO FINAL")
    print("=" * 60)
    
    main_path_disruptions = df_disruption[
        df_disruption["patent"].isin(main_path)
    ][["patent", "year", "d_prime", "ni", "nj", "nk"]].sort_values("year")
    
    print("\nNós do caminho principal:")
    print(main_path_disruptions.to_string(index=False))

    results = {
        "G_original"    : G,
        "G_contracted"  : G_contracted,
        "G_critical"    : G_critical,
        "df_disruption" : df_disruption,
        "df_annual"     : df_annual,
        "edge_weights"  : edge_weights,
        "main_path"     : main_path,
        "critical_path" : critical_path,
        "year_map"      : year_map,
    }
    return results

In [ ]:
# =============================================================================
# UTILITÁRIOS ADICIONAIS
# =============================================================================

def export_results(results: dict, output_prefix: str = "disruptive_path"):
    """Exporta todos os resultados para Excel."""
    with pd.ExcelWriter(f"{output_prefix}_results.xlsx", engine="openpyxl") as writer:
        results["df_disruption"].to_excel(writer, sheet_name="Disruption_Degree", index=False)
        results["df_annual"].to_excel(writer, sheet_name="Annual_Analysis", index=False)
        
        # Caminho principal
        mp = results["main_path"]
        df_mp = pd.DataFrame({
            "order": range(1, len(mp) + 1),
            "patent": mp,
            "year": [results["year_map"].get(n, "?") for n in mp],
        })
        d_map = dict(zip(results["df_disruption"]["patent"],
                         results["df_disruption"]["d_prime"]))
        df_mp["d_prime"] = df_mp["patent"].map(d_map)
        df_mp.to_excel(writer, sheet_name="Main_Path", index=False)
        
        # Arestas com pesos SPLC
        df_edges = pd.DataFrame([
            {"source": u, "target": v, "splc_weight": w}
            for (u, v), w in results["edge_weights"].items()
        ]).sort_values("splc_weight", ascending=False)
        df_edges.to_excel(writer, sheet_name="SPLC_Weights", index=False)
    
    print(f"[Export] Resultados salvos em: {output_prefix}_results.xlsx")


def compute_network_statistics(G: nx.DiGraph, 
                                df_disruption: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula estatísticas de rede por nó para análise adicional.
    Útil para identificar patentes "hub" vs patentes na fronteira disruptiva.
    """
    stats = []
    d_map = dict(zip(df_disruption["patent"], df_disruption["d_prime"]))
    
    for n in G.nodes():
        stats.append({
            "patent"         : n,
            "in_degree"      : G.in_degree(n),
            "out_degree"     : G.out_degree(n),
            "d_prime"        : d_map.get(n, np.nan),
            "betweenness"    : nx.betweenness_centrality(G).get(n, 0),
            "pagerank"       : nx.pagerank(G).get(n, 0),
        })
    
    return pd.DataFrame(stats).sort_values("pagerank", ascending=False)

In [ ]:
# =============================================================================
# INSPEÇÃO DOS DADOS BIGQUERY (execute antes do pipeline completo)
# =============================================================================

import os, pathlib

WORKSPACE = pathlib.Path("/home/vcunha/pen/SI/ic-esteban")
META_PATH = str(WORKSPACE / "bq-results-20260518-131545-1779110232837.json")
NET_PATH  = str(WORKSPACE / "bq-results-20260518-131903-1779110375248.json")

print("=" * 55)
print("INSPEÇÃO DOS ARQUIVOS BIGQUERY")
print("=" * 55)

# --- Metadados ---
df_meta_inspect = pd.read_json(META_PATH, lines=True, dtype={"priority_date": str})
df_meta_inspect['year'] = pd.to_datetime(
    df_meta_inspect['priority_date'].astype(str), format='%Y%m%d', errors='coerce'
).dt.year

print(f"\n[Metadados] {len(df_meta_inspect)} patentes focais")
print(f"  Colunas: {list(df_meta_inspect.columns)}")
print(f"  Anos: {int(df_meta_inspect['year'].min())} – {int(df_meta_inspect['year'].max())}")
print(f"  Amostra:")
print(df_meta_inspect[['publication_number','year','title']].head(5).to_string(index=False))

# --- Rede ---
df_net_inspect = pd.read_json(NET_PATH, lines=True, dtype={"citing_date": str})
df_net_inspect['citing_year'] = pd.to_datetime(
    df_net_inspect['citing_date'].astype(str), format='%Y%m%d', errors='coerce'
).dt.year

print(f"\n[Rede] {len(df_net_inspect)} pares de citação")
print(f"  Colunas: {list(df_net_inspect.columns)}")

focal_set_insp = set(df_meta_inspect['publication_number'])
back_insp = df_net_inspect[df_net_inspect['citing_patent'].isin(focal_set_insp)]
fwd_insp  = df_net_inspect[df_net_inspect['cited_patent'].isin(focal_set_insp)]
print(f"  Backward (focal cita algo): {len(back_insp)}")
print(f"  Forward  (algo cita focal): {len(fwd_insp)}")

print(f"\n  Pares únicos citantes (não focais): {df_net_inspect['citing_patent'].nunique()}")
print(f"  Pares únicos citados  (focais BR):   {df_net_inspect['cited_patent'].nunique()}")
print(f"\n  Amostra dos pares:")
print(df_net_inspect[['citing_patent','citing_year','cited_patent']].head(8).to_string(index=False))

focais_com_forward = set(df_net_inspect['cited_patent']) & focal_set_insp
print(f"\n  Focais com ≥1 citação forward na rede: {len(focais_com_forward)} / {len(focal_set_insp)}")


INSPEÇÃO DOS ARQUIVOS BIGQUERY


FileNotFoundError: File /home/vcunha/pen/SI/ic-esteban/bq-results-20260518-131545-1779110232837.json does not exist

In [ ]:
# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================
# Nota: Este notebook usa um kernel Jupyter remoto (sandbox sem acesso ao
# filesystem local). Para rodar o pipeline com os dados reais do BigQuery,
# execute diretamente via terminal:
#
#   python3 code/run_pipeline.py
#
# As figuras são salvas automaticamente em relatorio/img/.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

print("Executando run_pipeline.py via subprocess...")
result = subprocess.run(
    [sys.executable, "/home/vcunha/pen/SI/ic-esteban/code/run_pipeline.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("[ERRO]", result.stderr[-2000:])
    # Fallback: rodar pipeline sintético diretamente neste kernel
    print("\nFallback: executando pipeline sintético neste kernel...")
    results_syn = run_full_pipeline(
        data_source="synthetic",
        disruption_threshold=0.25,
        min_citations=1,
        save_figures=False,
    )
    print("\n[OK] Pipeline sintético concluído.")
else:
    print("[OK] Pipeline concluído. Figuras em relatorio/img/")
